In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False  # 让图表里的负号正常显示


In [2]:
file_path = r"../数据/全体A股.csv"  # 你刚刚导出的文件路径
df_all = pd.read_csv(file_path)

# 转换日期
df_all['trade_date'] = pd.to_datetime(df_all['trade_date'])

# 按股票代码 + 日期排序
df_all = df_all.sort_values(['ts_code','trade_date']).reset_index(drop=True)

import statsmodels.api as sm#线性回归工具

def neutralize_return_daily(df):
    """
    每天截面中性化：
    对数收益率 log_return 剔除 行业 + 市值 的影响
    输出：纯净收益率 log_return_neutral
    """
    def _neutralize(group):
        # 1. 市值取 log
        log_mv = np.log(group['market_cap'])

        # 2. 行业转哑变量，每一个行业赋值，便于识别
        ind_dummies = pd.get_dummies(group['industry'], drop_first=True)

        # 3. 构造回归 X
        X = pd.concat([ind_dummies, log_mv.rename('log_mv')], axis=1)
        X = sm.add_constant(X)

        # 4. 回归原始收益率
        y = group['log_return']

        try:
            # 残差 = 纯净收益率
            res = sm.OLS(y, X).fit().resid
        except:
            res = y  # 回归失败就用原值

        group['log_return_neutral'] = res
        return group

    # 按【每天】做截面中性化（最标准）
    return df.groupby('trade_date', group_keys=False).apply(_neutralize)

# 先计算原始 log_return
df_all['log_return'] = np.log(df_all['close'] / df_all['close'].shift(1))

# 执行中性化
df_all = neutralize_return_daily(df_all)


In [3]:

# 按股票分组，存入字典
all_stocks = {}
for code, group in df_all.groupby('ts_code'):
    df = group.copy().reset_index(drop=True)
    
    # 计算对数收益率
    df['log_return'] = np.log(df['close'] / df['close'].shift(1))
    
    # 涨跌幅（直接用表中的 pct_chg）
    df['pct_return'] = df['pct_chg']
    
    all_stocks[code] = df
    print(f"加载 {code}: {len(df)} 条记录 | 日期范围 {df['trade_date'].min().date()} ~ {df['trade_date'].max().date()}")

print("\n✅ 全部股票加载完成！总股票数：", len(all_stocks))

加载 000001.SZ: 2427 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000002.SZ: 2336 条记录 | 日期范围 2016-07-04 ~ 2026-02-27
加载 000004.SZ: 2224 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000005.SZ: 1942 条记录 | 日期范围 2016-03-01 ~ 2024-03-05
加载 000006.SZ: 2304 条记录 | 日期范围 2016-03-02 ~ 2026-02-27
加载 000007.SZ: 2082 条记录 | 日期范围 2016-07-15 ~ 2026-02-27
加载 000008.SZ: 2349 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000009.SZ: 2423 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000010.SZ: 2401 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000011.SZ: 2426 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000012.SZ: 2428 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000014.SZ: 2426 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000016.SZ: 2376 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000017.SZ: 2375 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000018.SZ: 891 条记录 | 日期范围 2016-03-01 ~ 2020-01-06
加载 000019.SZ: 2266 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000020.SZ: 2360 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000021.SZ: 2423 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000022.SZ: 531 条记录 | 日期范围 

In [4]:
#量化分析每只股票的噪声特征
def analyze_noise_characteristics(stocks_dict):
    results = []
    
    for code, df in stocks_dict.items():
        returns = df['log_return_neutral'].dropna()
        
        # 1. 基础统计
        std = returns.std()# 波动率 = 噪声强度
        skew = returns.skew()# 偏度 = 涨跌不对称性
        kurt = returns.kurtosis()# 峰度 = 极端值多少
        
        # 2. 自相关性（衡量噪声的持续性）
        from scipy.stats import pearsonr
        autocorr_1 = returns.autocorr(lag=1) # 今日与昨日相关性
        autocorr_5 = returns.autocorr(lag=5)# 今日与5日前相关性
        #接近 0 = 市场有效、随机游走、纯噪声
        #不等于 0 = 有趋势、有规律
        # 3. 波动率聚集特征（用GARCH类型效应，简化为滚动标准差变异系数）
        rolling_std = returns.rolling(window=20).std().dropna()
        vol_clustering = rolling_std.std() / rolling_std.mean()  # CV越大，波动聚集越明显
        
        # 4. 极端值频率（超过3倍标准差的交易日占比）
        extreme_ratio = (abs(returns) > 3 * std).sum() / len(returns)
        
        # 5. 信息比率的倒数（噪声/信号）
        noise_to_signal = returns.std() / abs(returns.mean()) if returns.mean() != 0 else np.inf
        
        results.append({
            'code': code,
            '收益率标准差（波动率）': std,
            '偏度': skew,
            '峰度': kurt,
            '一阶自相关系数': autocorr_1,
            '五阶自相关系数': autocorr_5,
            '波动率聚集变异系数': vol_clustering,
            '极端值比例': extreme_ratio,
            '噪声信号比': noise_to_signal
        })
    
    return pd.DataFrame(results)

# 执行分析
noise_analysis = analyze_noise_characteristics(all_stocks)
noise_analysis = noise_analysis.sort_values('收益率标准差（波动率）', ascending=False)

print("\n=== 噪声特征分析结果 ===")
print(noise_analysis.to_string(index=False))

# 保存结果
noise_analysis.to_csv('../数据/噪声特征.csv', index=False, encoding='utf-8-sig')



=== 噪声特征分析结果 ===
     code  收益率标准差（波动率）         偏度          峰度   一阶自相关系数       五阶自相关系数  波动率聚集变异系数    极端值比例        噪声信号比
920680.BJ     1.074350  -1.965274    3.884620 -0.846281           NaN        NaN 0.000000 1.494533e+00
688816.SH     0.703517  -2.235244    4.997052  0.655254           NaN        NaN 0.000000 2.199727e+00
688785.SH     0.700351   3.843064   14.834854  0.591124  7.028841e-02        NaN 0.066667 4.101409e+00
920045.BJ     0.600236   5.503594   30.504500  0.148002 -1.182293e-02   1.786129 0.032258 5.340765e+00
688795.SH     0.467533   6.917134   48.196735  0.189386 -5.398514e-02   2.463363 0.020408 6.976176e+00
920119.BJ     0.410372   3.704881   13.801226 -0.211344 -7.457188e-02        NaN 0.071429 3.780433e+00
920180.BJ     0.387109  -2.625612    6.919147  0.650623  1.000000e+00        NaN 0.000000 2.565935e+00
920166.BJ     0.375057   1.703968         NaN  1.000000           NaN        NaN 0.000000 1.803404e+00
001220.SZ     0.374197   3.299149   10.916513 -0.701918